In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data=pd.read_csv("C:\Projects\coustmer_churn\data\Telco-Customer-Churn.csv")

In [3]:
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
data.shape

(7043, 21)

In [5]:
data.info()#no missing values

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [6]:
data['TotalCharges']=pd.to_numeric(data['TotalCharges'],errors='coerce')
data['TotalCharges'].isna().sum()

np.int64(11)

In [7]:
data.dropna(inplace=True)

print(data.isna().any())

customerID          False
gender              False
SeniorCitizen       False
Partner             False
Dependents          False
tenure              False
PhoneService        False
MultipleLines       False
InternetService     False
OnlineSecurity      False
OnlineBackup        False
DeviceProtection    False
TechSupport         False
StreamingTV         False
StreamingMovies     False
Contract            False
PaperlessBilling    False
PaymentMethod       False
MonthlyCharges      False
TotalCharges        False
Churn               False
dtype: bool


In [8]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(data.drop(columns=['Churn']),data.iloc[:,-1],test_size=0.2,random_state=67)

In [9]:
y_train.head()

3526    Yes
1837    Yes
1334    Yes
1906     No
3994    Yes
Name: Churn, dtype: str

In [10]:
y_train.value_counts(normalize=True)#unbalanced data

Churn
No     0.735111
Yes    0.264889
Name: proportion, dtype: float64

In [11]:
x_train=x_train.drop(columns='customerID').reset_index(drop=True)

In [12]:
x_test=x_test.drop(columns='customerID').reset_index(drop=True)

In [13]:
nominal_data=[]
numeric_data=[]
for column in x_train.columns:
    if x_train[column].dtype=='str':
        nominal_data.append(column)
    if x_train[column].dtype=='float64' or x_train[column].dtype=='int64':
        numeric_data.append(column)
numeric_data.remove('SeniorCitizen')

In [14]:
for column in nominal_data:
    print(x_train[column].value_counts())

gender
Male      2839
Female    2786
Name: count, dtype: int64
Partner
No     2926
Yes    2699
Name: count, dtype: int64
Dependents
No     3965
Yes    1660
Name: count, dtype: int64
PhoneService
Yes    5087
No      538
Name: count, dtype: int64
MultipleLines
No                  2716
Yes                 2371
No phone service     538
Name: count, dtype: int64
InternetService
Fiber optic    2475
DSL            1952
No             1198
Name: count, dtype: int64
OnlineSecurity
No                     2809
Yes                    1618
No internet service    1198
Name: count, dtype: int64
OnlineBackup
No                     2499
Yes                    1928
No internet service    1198
Name: count, dtype: int64
DeviceProtection
No                     2473
Yes                    1954
No internet service    1198
Name: count, dtype: int64
TechSupport
No                     2807
Yes                    1620
No internet service    1198
Name: count, dtype: int64
StreamingTV
No                     2270
Y

In [15]:
x_train[numeric_data].describe()

,tenure,MonthlyCharges,TotalCharges
count,5625.000000,5625.000000,5625.000000
mean,32.220978,64.888969,2273.802382
std,24.583032,29.952540,2260.796493
min,1.000000,18.250000,18.800000
25%,8.000000,36.450000,385.900000
50%,29.000000,70.350000,1402.250000
75%,55.000000,89.850000,3782.400000
max,72.000000,118.750000,8684.800000


In [16]:
q1=x_train[numeric_data].quantile(0.25)
q3=x_train[numeric_data].quantile(0.75)
iqr=q3-q1
lb=q1 - 1.5*iqr
ub=q3 + 1.5*iqr

In [17]:
outliers = x_train[(x_train[numeric_data] < lb) | (x_train[numeric_data] > ub)]

In [18]:
outliers[numeric_data].isna().sum()# no outliers

tenure            5625
MonthlyCharges    5625
TotalCharges      5625
dtype: int64

In [19]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [20]:
transformer=ColumnTransformer([('tnf2',StandardScaler(),numeric_data),
                               ('tnf3',OneHotEncoder(drop='first'),nominal_data)],remainder='passthrough')

In [21]:
transformer.fit(x_train)
columns=transformer.get_feature_names_out()

In [22]:
for column in columns:
    print(column)

tnf2__tenure
tnf2__MonthlyCharges
tnf2__TotalCharges
tnf3__gender_Male
tnf3__Partner_Yes
tnf3__Dependents_Yes
tnf3__PhoneService_Yes
tnf3__MultipleLines_No phone service
tnf3__MultipleLines_Yes
tnf3__InternetService_Fiber optic
tnf3__InternetService_No
tnf3__OnlineSecurity_No internet service
tnf3__OnlineSecurity_Yes
tnf3__OnlineBackup_No internet service
tnf3__OnlineBackup_Yes
tnf3__DeviceProtection_No internet service
tnf3__DeviceProtection_Yes
tnf3__TechSupport_No internet service
tnf3__TechSupport_Yes
tnf3__StreamingTV_No internet service
tnf3__StreamingTV_Yes
tnf3__StreamingMovies_No internet service
tnf3__StreamingMovies_Yes
tnf3__Contract_One year
tnf3__Contract_Two year
tnf3__PaperlessBilling_Yes
tnf3__PaymentMethod_Credit card (automatic)
tnf3__PaymentMethod_Electronic check
tnf3__PaymentMethod_Mailed check
remainder__SeniorCitizen


In [23]:
x_train_new=pd.DataFrame(transformer.fit_transform(x_train),columns=columns)

In [24]:
x_test_new=pd.DataFrame(transformer.fit_transform(x_test),columns=columns)

In [25]:
x_train_new.head()

,tnf2__tenure,tnf2__MonthlyCharges,tnf2__TotalCharges,tnf3__gender_Male,tnf3__Partner_Yes,tnf3__Dependents_Yes,tnf3__PhoneService_Yes,tnf3__MultipleLines_No phone service,tnf3__MultipleLines_Yes,tnf3__InternetService_Fiber optic,...,tnf3__StreamingTV_Yes,tnf3__StreamingMovies_No internet service,tnf3__StreamingMovies_Yes,tnf3__Contract_One year,tnf3__Contract_Two year,tnf3__PaperlessBilling_Yes,tnf3__PaymentMethod_Credit card (automatic),tnf3__PaymentMethod_Electronic check,tnf3__PaymentMethod_Mailed check,remainder__SeniorCitizen
0,-1.107406,-0.485443,-0.900892,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,-1.270134,-1.497134,-0.996973,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-1.270134,-1.325180,-0.994695,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
3,0.316467,-0.964577,-0.394101,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0.886016,1.222410,1.371009,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0


In [26]:
x_test_new.head()

,tnf2__tenure,tnf2__MonthlyCharges,tnf2__TotalCharges,tnf3__gender_Male,tnf3__Partner_Yes,tnf3__Dependents_Yes,tnf3__PhoneService_Yes,tnf3__MultipleLines_No phone service,tnf3__MultipleLines_Yes,tnf3__InternetService_Fiber optic,...,tnf3__StreamingTV_Yes,tnf3__StreamingMovies_No internet service,tnf3__StreamingMovies_Yes,tnf3__Contract_One year,tnf3__Contract_Two year,tnf3__PaperlessBilling_Yes,tnf3__PaymentMethod_Credit card (automatic),tnf3__PaymentMethod_Electronic check,tnf3__PaymentMethod_Mailed check,remainder__SeniorCitizen
0,-0.993741,-0.497711,-0.818029,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1,1.385534,1.338240,2.058564,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0
2,-1.280895,-1.449989,-0.996471,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,-1.321917,-1.469589,-1.005117,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
4,-0.993741,0.018446,-0.724081,0.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [27]:
#using knn for imbalenced data
from imblearn.over_sampling import SMOTE
bal_data=SMOTE(random_state=67)
x_train_res,y_train_res=bal_data.fit_resample(x_train_new,y_train)

In [36]:
x_train_res.shape

(8270, 30)

In [37]:
y_train_res.value_counts(normalize=True)

Churn
Yes    0.5
No     0.5
Name: proportion, dtype: float64

In [39]:
feature=LabelEncoder()
y_train_new=feature.fit_transform(y_train_res)
y_test_new=feature.transform(y_test)

In [40]:
y_train_new=pd.Series(y_train_new)
y_train_new.name='Chrun'
y_test_new=pd.Series(y_test_new)
y_test_new.name='Chrun'

In [42]:
y_train_new.value_counts(normalize=True)

Chrun
1    0.5
0    0.5
Name: proportion, dtype: float64

In [44]:
y_train_new.to_csv(r"C:\Projects\coustmer_churn\new data\y_train")
x_train_res.to_csv(r"C:\Projects\coustmer_churn\x_train")
y_test_new.to_csv(r"C:\Projects\coustmer_churn\new data\y_test")
x_test_new.to_csv(r"C:\Projects\coustmer_churn\new data\x_test")